In [1]:
%pip install pgmpy

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.4 MB 1.8 MB/s eta 0:00:02
   ------------------------------ --------- 1.8/2.4 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 4.8 MB/s  0:00:00
   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   ---------------------------------------- 857.0/857.0 kB 7.4 MB/s  0:00:00
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ------------------------------------- -- 3.7/3.9 MB 17.8 MB/s eta 0:00:01
   ---------------------------------------- 3.9/3.9 MB 15.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 13.0 MB/s  0:00:00
   ---------------------------------------- 0.0/12.6 MB ?


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import warnings
warnings.filterwarnings("ignore")   # hide library notices

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

# 1) The graph: each pair is an arrow (parent, child)
model = DiscreteBayesianNetwork([
    ("rain", "maintenance"),
    ("rain", "train"),
    ("maintenance", "train"),
    ("train", "appointment"),
])

# 2) P(Rain): no parents, so one column
rain = TabularCPD(
    variable="rain", variable_card=3,
    values=[[0.7],    # none
            [0.2],    # light
            [0.1]],   # heavy
    state_names={"rain": ["none", "light", "heavy"]})

# 3) P(Maintenance | Rain): one column for each value of rain
#                rain = none  light  heavy
maintenance = TabularCPD(
    variable="maintenance", variable_card=2,
    values=[[0.4, 0.2, 0.1],    # yes
            [0.6, 0.8, 0.9]],   # no
    evidence=["rain"], evidence_card=[3],
    state_names={"maintenance": ["yes", "no"],
                 "rain": ["none", "light", "heavy"]})

# 4) P(Train | Rain, Maintenance): one column for each (rain, maintenance) pair
#     (none,yes) (none,no) (light,yes) (light,no) (heavy,yes) (heavy,no)
train = TabularCPD(
    variable="train", variable_card=2,
    values=[[0.8, 0.9, 0.6, 0.7, 0.4, 0.5],    # on time
            [0.2, 0.1, 0.4, 0.3, 0.6, 0.5]],   # delayed
    evidence=["rain", "maintenance"], evidence_card=[3, 2],
    state_names={"train": ["on time", "delayed"],
                 "rain": ["none", "light", "heavy"],
                 "maintenance": ["yes", "no"]})

# 5) P(Appointment | Train): one column for each value of train
#                train = on time  delayed
appointment = TabularCPD(
    variable="appointment", variable_card=2,
    values=[[0.9, 0.6],    # attend
            [0.1, 0.4]],   # miss
    evidence=["train"], evidence_card=[2],
    state_names={"appointment": ["attend", "miss"],
                 "train": ["on time", "delayed"]})

# 6) Put the tables into the graph and check them
model.add_cpds(rain, maintenance, train, appointment)
print("Model is correct:", model.check_model())

Model is correct: True


In [2]:
# P(r, m, t, a) = P(r) x P(m | r) x P(t | r, m) x P(a | t)
p = (rain.get_value(rain="none")
     * maintenance.get_value(maintenance="no", rain="none")
     * train.get_value(train="on time", rain="none", maintenance="no")
     * appointment.get_value(appointment="attend", train="on time"))
print(round(p, 4))

0.3402


In [3]:
p = (rain.get_value(rain="heavy")
     * maintenance.get_value(maintenance="yes", rain="heavy")
     * train.get_value(train="delayed", rain="heavy", maintenance="yes")
     * appointment.get_value(appointment="miss", train="delayed"))
print(round(p, 4))

0.0024


In [4]:
from pgmpy.inference import VariableElimination
inference = VariableElimination(model)

def show(evidence):
    """Print P(X | evidence) for every node X that we did not observe."""
    for node in ["rain", "maintenance", "train", "appointment"]:
        if node in evidence:
            print(f"{node}: {evidence[node]}   (observed)")
            continue
        result = inference.query([node], evidence=evidence, show_progress=False)
        print(node)
        for value, prob in zip(result.state_names[node], result.values):
            print(f"    {value}: {prob:.4f}")

show({"train": "delayed"})

rain
    none: 0.4601
    light: 0.3005
    heavy: 0.2394
maintenance
    yes: 0.3662
    no: 0.6338
train: delayed   (observed)
appointment
    attend: 0.6000
    miss: 0.4000


In [5]:
show({"appointment": "miss"})

rain
    none: 0.6065
    light: 0.2392
    heavy: 0.1544
maintenance
    yes: 0.3441
    no: 0.6559
train
    on time: 0.4802
    delayed: 0.5198
appointment: miss   (observed)


In [6]:
show({"train": "delayed", "rain": "heavy"})

rain: heavy   (observed)
maintenance
    yes: 0.1176
    no: 0.8824
train: delayed   (observed)
appointment
    attend: 0.6000
    miss: 0.4000
